In [ ]:
# !pip install --upgrade google-play-scraper
# !pip install pyqt5
# !pip install --upgrade beautifulsoup4
# !pip install pyqtwebengine
# !pip install pandas
# !pip install selenium

In [ ]:
# from google_play_scraper import collection
# from google_play_scraper.constants import Collection, Category

import time
import json
import pandas as pd
import requests
from bs4 import BeautifulSoup
from google_play_scraper import app, permissions



#Identified risk categories in each app's permissions
risk_categories = {
    "overlay": ["draw over other apps"],
    "background": ["run at startup", "prevent device from sleeping"],
    "bluetooth": ["pair with Bluetooth devices", "access Bluetooth settings"],
    "network": ["full network access", "view network connections"]
}


In [ ]:
def collect_app_data():
    error_apps = {}

    with open("family_apps_strict_enriched.json", "r", encoding="utf-8") as f:
        data = json.load(f)

    df = pd.json_normalize(data)

    flag_dict = {}
    seen_apps = set()

    for _, row in df.iterrows():
        app_id = row.get("appId")

        if not app_id or app_id in seen_apps:
            continue
        seen_apps.add(app_id)

        try:
            perm = permissions(app_id)
            url = row.get("privacyPolicy")
            title = row.get("title")
            genre = row.get("genre")

            if not url:
                continue

            response = requests.get(url, timeout=(3, 5))
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")

            for tag in soup(["script", "style", "noscript"]):
                tag.decompose()

            policy_text = soup.get_text("\n").lower()
            other_permissions = perm.get("Other", [])

            flag_dict[app_id] = {
                "title": title,
                "genre": genre,
                "permissions": other_permissions,
                "policy_text": policy_text,
            }

        except requests.RequestException as e:
            print(f"Request error for {app_id}: {e}")
            error_apps[app_id] = str(e)
        except Exception as e:
            print(f"Error fetching {app_id}: {e}")
            error_apps[app_id] = str(e)

    if error_apps:
        print(f"Skipped {len(error_apps)} apps due to fetch/parse errors.")

    return flag_dict

In [ ]:
import json
import os

CACHE_FILE = "app_data_cache.json"
def load_flag_dict():
    if os.path.exists(CACHE_FILE):
        print("Loading from cache...")
        with open(CACHE_FILE, "r") as f:
            return json.load(f)
    
    print("Fetching app data...")
    flag_dict = collect_app_data()
    
    with open(CACHE_FILE, "w") as f:
        json.dump(flag_dict, f, indent=2)
    
    return flag_dict

In [ ]:
# Track apps that request at least one Bluetooth-related permission
bluetooth_apps = []

# Track apps that request network-access-related permission
network_apps = []

# Call data collection function
flag_dict = load_flag_dict()

bluetooth_risk_perms = set(risk_categories.get("bluetooth", []))
network_risk_perms = set(risk_categories.get("network", []))

# For each app, check permission overlap with risk categories
for app_id, data in flag_dict.items():
    perms = set(data.get("permissions", []))
    title = data.get("title") or app_id

    if perms & bluetooth_risk_perms:
        bluetooth_apps.append(title)

    if perms & network_risk_perms:
        network_apps.append(title)

total_apps = len(flag_dict)
bluetooth_rate = len(bluetooth_apps) / total_apps if total_apps else 0.0
network_rate = len(network_apps) / total_apps if total_apps else 0.0

print(f"Apps that ask for Bluetooth permission: {len(bluetooth_apps)} / {total_apps} = {bluetooth_rate:.4f}")
print(f"Apps that ask for Network permission:   {len(network_apps)} / {total_apps} = {network_rate:.4f}")

print("\nBluetooth Permission Rate:", round(bluetooth_rate * 100, 2), "%")
print("Network Permission Rate:", round(network_rate * 100, 2), "%")

## Cell Below may be adapted or deleted in the future. I'll keep it here for now. Right now it just checks for the keyword "bluetooth", but the cell below this one identifies key bluetooth phrases not just the word bluetooth.

In [ ]:

#list of undisclosed apps (Bluetooth)
bluetooth_undisclosed = []

network_undisclosed = []


network_rate = len(network_apps) / len(flag_dict)

print(f"Apps that ask for Network permission(s): {len(network_apps)} / {len(flag_dict)} = {network_rate}")

print(f"\nNetwork Permission Rate: {(network_rate * 100):.2f}%")


#create perms and policy variables
for app_id, data in flag_dict.items():
    perms = data["permissions"]
    policy = data["policy_text"]


    #Boolean check for bluetooth risk and if it mentions Bluetooth in the policy.
    #If not, add it to the undisclosed apps list
    has_bt = any(p in risk_categories["bluetooth"] for p in perms)
    mentions_bt = "bluetooth" in policy

    has_net = any(p in risk_categories["network"] for p in perms)
    mentions_net = "network" in policy

    if has_bt and not mentions_bt:
        bluetooth_undisclosed.append(data["title"])

    if has_net and not mentions_net:
      network_apps.append(data["title"])


print("Bluetooth Non-Disclosure Cases:")
print(f"{len(bluetooth_undisclosed)} / {len(flag_dict)}")

print("\nNetwork Non-Disclosure Cases:")
print(f"{len(network_undisclosed)} / {len(flag_dict)}")







In [ ]:
#NLP and BERT integration
#Bluetooth Disclosure = policy mentions scanning, connecting, pairing,
#or communicating with nearby devices wirelessly, regardless of the keyword "Bluetooth" being mentioned

BLUETOOTH_TERMS = [
  "bluetooth", "ble", "bluetooth low energy",
  "nearby devices", "device discovery", "scan for devices",
  "scanning for devices", "pairing", "paired",
  "ibeacon", "eddystone", "beacon"
]

bt_disclosed = []

bluetooth_undisclosed = []
for app_id in bluetooth_apps:
    data = flag_dict.get(app_id, {})
    policy = (data.get("policy_text", "") or "").lower()

    if any(term in policy for term in BLUETOOTH_TERMS):
        bt_disclosed.append(app_id)
    else:
        bluetooth_undisclosed.append(app_id)

print("apps:", len(bluetooth_apps))
print("undisclosed:", len(bluetooth_undisclosed))
print("Disclosed:", len(bt_disclosed))






In [ ]:
!pip install selenium webdriver-manager lxml beautifulsoup4 requests


In [16]:
def print_all_sections(driver):
    spans = driver.find_elements(By.XPATH, "//span[@jsname]")
    seen = set()
    for span in spans:
        text = span.text.strip()
        if text and text not in seen:
            seen.add(text)
            print(f"   FOUND: '{text}'")

In [17]:

# Google Play Family Apps Scraper
# Uses webdriver-manager to automatically download ChromeDriver — no manual setup needed.
# Install dependencies: pip install selenium webdriver-manager

%pip install -q webdriver-manager

import time
import csv
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from webdriver_manager.chrome import ChromeDriverManager

# ==========================================
# CONFIGURATION
# ==========================================
OUTPUT_CSV = "all_categories_multiple_total.csv"

# Set to True to run Chrome in the background (no browser window)
HEADLESS = False

# LIST OF URLS TO SCRAPE
TARGET_URLS = [
    'https://play.google.com/store/apps/category/FAMILY',
    'https://play.google.com/store/apps/category/FAMILY?age=AGE_RANGE1',
    'https://play.google.com/store/apps/category/FAMILY?age=AGE_RANGE2',
    'https://play.google.com/store/apps/category/FAMILY?age=AGE_RANGE3'
]

# The sections to look for on EACH page
TARGET_SECTIONS = [
    "Recommended for you",
    "New & updated",
    "Love for the whole family",
    "Family apps",
    "Entertainment",
    "Simulation games",
    "Offline games",
    "Games you might like",
    "Stylized games",
    "Cartoony games",
    "Casual games",
    "Educational apps",
    "Animal games",
    "Fun watch apps for kids",
    "Games for kids",
    "Apps for kids",
    "Enriching games",
    "Action games for kids"
]

# ==========================================
# SETUP DRIVER (auto-downloads ChromeDriver)
# ==========================================
options = webdriver.ChromeOptions()
if HEADLESS:
    options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")

service = ChromeService(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
actions = ActionChains(driver)

collected_apps = set()

# ==========================================
# SCRAPING FUNCTION
# ==========================================
def scrape_specific_section(driver, section_title):
    """
    Locates a specific section header, waits for UI to settle, then scrapes the carousel.
    """
    print(f"   [SECTION] Looking for: '{section_title}'...")

    try:
        section_header = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.XPATH, f"//span[text()='{section_title}']"))
        )

        driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", section_header)
        time.sleep(2)

        target_section = section_header.find_element(By.XPATH, "./ancestor::div[@class='fM1mj'][1] | ./ancestor::section")

        actions.move_to_element(target_section).perform()
        time.sleep(2)

        while True:
            apps = target_section.find_elements(By.XPATH, ".//a[contains(@href, '/store/apps/details?id=')]")

            for app in apps:
                href = app.get_attribute("href")
                if href and href not in collected_apps:
                    collected_apps.add(href)

            try:
                next_button = target_section.find_element(By.XPATH, ".//button[@aria-label='Scroll Next']")
                actions.move_to_element(next_button).perform()

                if next_button.is_displayed():
                    driver.execute_script("arguments[0].click();", next_button)
                    time.sleep(1.5)
                else:
                    break
            except NoSuchElementException:
                break
            except Exception as e:
                print(f"      -> Warning: Issue clicking next: {e}")
                break

        print(f"      -> Finished '{section_title}'.")

    except TimeoutException:
        print(f"      -> Skipped: Could not find section '{section_title}' on this URL.")
    except Exception as e:
        print(f"      -> Error processing '{section_title}': {e}")


# ==========================================
# MAIN EXECUTION
# ==========================================
try:
    for index, url in enumerate(TARGET_URLS):
        print(f"\n==========================================")
        print(f"PROCESSING URL {index + 1}/{len(TARGET_URLS)}: {url}")
        print(f"==========================================")

        driver.get(url)
        time.sleep(5)
        print_all_sections(driver)


        for section in TARGET_SECTIONS:
            scrape_specific_section(driver, section)
            time.sleep(1)

    # ==========================================
    # SAVE TO CSV
    # ==========================================
    print(f"\n==========================================")
    print(f"ALL DONE. Total unique apps collected: {len(collected_apps)}")

    with open(OUTPUT_CSV, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(["App URL", "Package ID"])

        for url in collected_apps:
            try:
                pkg = url.split("id=")[1].split("&")[0]
            except Exception:
                pkg = "Unknown"
            writer.writerow([url, pkg])

    print(f"Data saved to {OUTPUT_CSV}")

except Exception as e:
    print(f"CRITICAL ERROR: {e}")

finally:
    driver.quit()



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.

PROCESSING URL 1/4: https://play.google.com/store/apps/category/FAMILY
   FOUND: 'Games
Apps
Movies & TV
Books
Kids'
   FOUND: 'Games'
   FOUND: 'Apps'
   FOUND: 'Movies & TV'
   FOUND: 'Books'
   FOUND: 'Kids'
   FOUND: 'All ages up to 12'
   FOUND: 'Ages up to 5'
   FOUND: 'Ages 6-8'
   FOUND: 'Ages 9-12'
   FOUND: 'Install'
   [SECTION] Looking for: 'Recommended for you'...
      -> Skipped: Could not find section 'Recommended for you' on this URL.
   [SECTION] Looking for: 'New & updated'...
      -> Finished 'New & updated'.
   [SECTION] Looking for: 'Love for the whole family'...
      -> Skipped: Could not find section 'Love for the whole family' on this URL.
   [SECTION] Looking for: 'Family apps'...
      -> Skipped: Could not find section 'Family apps' on this URL.
   [SECTION] Looking for: 'Entertainment'...
      -> Skipped: Could not find section 'Entertainment' on this URL.
   [SECTION] Looking for: 'Simul

# Task
Load the app data from "family_apps_strict_enriched.json" into a pandas DataFrame. Group the data by 'genre' and sample a total of 50-100 apps, ensuring a diverse mix of categories (e.g., Education, Games). Select the columns 'title' (rename to App Name), 'appId' (rename to Package ID), and 'privacyPolicy' (rename to Privacy Policy URL), then save this sampled dataframe to a CSV file named "sampled_apps_list.csv". Finally, confirm the file creation and display the first few rows of the sampled data.

## Load App Data

### Subtask:
Load the app data from 'family_apps_strict_enriched.json' into a pandas DataFrame.


**Reasoning**:
Load the JSON data into a pandas DataFrame as requested by the subtask instructions.



In [ ]:
import json
import pandas as pd

# Open the file and load the data
with open("family_apps_strict_enriched.json", "r") as f:
    data = json.load(f)

# Convert to DataFrame
df = pd.json_normalize(data)

# Display the first 5 rows and the shape
print("Shape of the DataFrame:", df.shape)
df.head()